# Volatility targeting con bandas

> ¿Pueden las bandas de no negociación reducir los costos de volatility targeting sin deteriorar significativamente el control del riesgo?

La notebook contendrá cinco bloques, desarrollados progresivamente:

| Bloque | Contenido                                              |
| ------ | ------------------------------------------------------ |
| 01     | Datos reales de SPY y validación                       |
| 02     | Pronósticos EWMA $ \lambda = 0.94 $                   |
| 03     | Exposición objetivo y políticas P0, P1 y P2            |
| 04     | Backtest con ejecución en apertura $ (t+1) $ y costos  |
| 05     | Evaluación estadística y económica frente a benchmarks |

Parámetros iniciales: volatilidad objetivo del 12%, exposición máxima 1× y bandas de 0, 5 y 10 puntos porcentuales.



## Bloque 01 — Datos de SPY

Usamos la serie diaria **Gold** aprobada por el quality gate. Revisamos cuatro cosas sencillas:

1. Cobertura temporal.
2. Fechas únicas y ordenadas.
3. Precios de apertura, máximo, mínimo, cierre y cierre ajustado.
4. Dividendos y splits, para saber qué ajustes existen.

In [3]:
from pathlib import Path

import polars as pl

def find_project_root() -> Path:
    # Funciona si Jupyter abre la notebook desde volatility_target o desde su carpeta padre.
    for folder in (Path.cwd(), *Path.cwd().parents):
        for candidate in (folder, folder / 'volatility_target'):
            if (candidate / 'data' / 'gold' / 'daily_series' / 'etf' / 'SPY.parquet').exists():
                return candidate
    raise FileNotFoundError('No encontré data/gold/daily_series/etf/SPY.parquet. Ejecuta primero el pipeline de SPY.')


project_root = find_project_root()
spy_path = project_root / 'data' / 'gold' / 'daily_series' / 'etf' / 'SPY.parquet'
spy = pl.read_parquet(spy_path)

print(f"Filas: {spy.height:,}")
print(f"Periodo: {spy['session_date'].min()} a {spy['session_date'].max()}")

spy.select([
    'session_date', 'open', 'close', 'adjusted_close',
    'dividend_amount', 'split_coefficient',
]).tail(5)

Filas: 2,227
Periodo: 2018-01-02 a 2026-11-10


session_date,open,close,adjusted_close,dividend_amount,split_coefficient
date,f64,f64,f64,f64,f64
2026-11-04,762.99,766.81,766.81,0.0,1.0
2026-11-05,765.47,733.0,733.0,0.0,1.0
2026-11-06,734.82,735.29,735.29,0.0,1.0
2026-11-09,741.95,752.69,752.69,0.0,1.0
2026-11-10,747.94,749.92,749.92,0.0,1.0


In [4]:
price_columns = ['open', 'high', 'low', 'close', 'adjusted_close']

checks = {
    'Fechas únicas': spy['session_date'].n_unique() == spy.height,
    'Fechas ordenadas': spy['session_date'].to_list() == sorted(spy['session_date'].to_list()),
    'Sin precios faltantes': sum(spy[column].null_count() for column in price_columns) == 0,
    'OHLC coherente': spy.select(
        ((pl.col('high') >= pl.max_horizontal('open', 'close', 'low')) &
         (pl.col('low') <= pl.min_horizontal('open', 'close', 'high'))).all()
    ).item(),
}

for name, passed in checks.items():
    print(f"{'✓' if passed else '✗'} {name}")

print(f"Sesiones con dividendo: {spy.filter(pl.col('dividend_amount') > 0).height}")
print(f"Sesiones con split: {spy.filter(pl.col('split_coefficient') != 1).height}")

✓ Fechas únicas
✓ Fechas ordenadas
✓ Sin precios faltantes
✓ OHLC coherente
Sesiones con dividendo: 35
Sesiones con split: 0


### Conclusión del Bloque 01

La serie Gold de SPY cubre del **2018-01-02** al **2026-11-10** y contiene **2,227** sesiones. La auditoría aprobó la serie: no hay fechas duplicadas, precios faltantes ni inconsistencias OHLC. Hay 35 sesiones con dividendos y ningún split en este periodo.

Por tanto, tenemos precios de apertura, cierre y cierre ajustado suficientes para continuar después con EWMA, sin implementar todavía ninguna política de rebalanceo.

In [5]:
spy

symbol_requested,session_date,open,high,low,close,adjusted_close,volume,dividend_amount,split_coefficient,provider_timezone,source,provider_last_refreshed,ingested_at_utc
str,date,f64,f64,f64,f64,f64,i64,f64,f64,str,str,str,"datetime[μs, UTC]"
"""SPY""",2018-01-02,267.84,268.81,267.4,268.77,235.20385,86655749,0.0,1.0,"""US/Eastern""","""ALPHA_VANTAGE""","""2026-11-10""",2026-11-11 01:35:59.228870 UTC
"""SPY""",2018-01-03,268.96,270.64,268.96,270.47,236.69154,90070416,0.0,1.0,"""US/Eastern""","""ALPHA_VANTAGE""","""2026-11-10""",2026-11-11 01:35:59.228870 UTC
"""SPY""",2018-01-04,271.2,272.16,270.54,271.61,237.689168,80636408,0.0,1.0,"""US/Eastern""","""ALPHA_VANTAGE""","""2026-11-10""",2026-11-11 01:35:59.228870 UTC
"""SPY""",2018-01-05,272.51,273.56,271.95,273.42,239.273121,83523995,0.0,1.0,"""US/Eastern""","""ALPHA_VANTAGE""","""2026-11-10""",2026-11-11 01:35:59.228870 UTC
"""SPY""",2018-01-08,273.31,274.1,272.98,273.92,239.710677,57319192,0.0,1.0,"""US/Eastern""","""ALPHA_VANTAGE""","""2026-11-10""",2026-11-11 01:35:59.228870 UTC
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""SPY""",2026-11-04,762.99,775.64,762.64,766.81,766.81,62126352,0.0,1.0,"""US/Eastern""","""ALPHA_VANTAGE""","""2026-11-10""",2026-11-11 01:35:59.228870 UTC
"""SPY""",2026-11-05,765.47,765.97,732.71,733.0,733.0,125892776,0.0,1.0,"""US/Eastern""","""ALPHA_VANTAGE""","""2026-11-10""",2026-11-11 01:35:59.228870 UTC
"""SPY""",2026-11-06,734.82,742.2104,734.58,735.29,735.29,58978818,0.0,1.0,"""US/Eastern""","""ALPHA_VANTAGE""","""2026-11-10""",2026-11-11 01:35:59.228870 UTC
